HAWKEYE — SIMCLR CONTRASTIVE PRE-TRAINING (Kaggle notebook)
============================================================
Self-supervised pre-training over per-account behavioural feature vectors.
Trains an encoder that produces 128-dim embeddings clustering similar
behaviour together — WITHOUT using fraud labels.

Why this matters
----------------
The 'zero-label cold-start' Round 1 claim. A bank with no historical fraud
labels can't use the supervised LightGBM blend on day one. SimCLR gives
them a useful representation from their own UNlabeled behaviour. After
pre-training, the bank's first 50 labeled incidents are enough to fine-tune
a logistic head on top.

Architecture
------------
- Input: 105-dim "feat_clean" vector per account (the LightGBM M1 feature set)
- Augmentations (per-anchor):
    A1: random feature dropout (mask 20% of features to 0)
    A2: gaussian noise (sigma=0.1 of feature std) on numeric features
- Encoder: 3-layer MLP (256 → 256 → 128), ELU, BatchNorm
- Projection head: 2-layer MLP (128 → 128 → 64) — used for NT-Xent loss only,
  DROPPED at inference (we use the encoder output as the embedding)
- Loss: NT-Xent (Normalized Temperature-scaled Cross Entropy), τ=0.5

INPUT  (Kaggle dataset attached): /kaggle/input/.../rbih-nfpc-phase-2/
        Plus: /kaggle/input/.../hawkeye_artifacts/account_feature_matrix.parquet
              (the output of ml-idea.ipynb — needed for the 105 feat_clean cols)
OUTPUT (downloaded after run):    /kaggle/working/hawkeye_simclr/
                                    ├── simclr_encoder.pt           (~1 MB)
                                    ├── simclr_embeddings.parquet   (~10 MB, 160k × 128)
                                    ├── feat_clean_scaler.json      (mean/std per feature)
                                    └── simclr_metadata.json        (eval AUC of LR head on embeds)

Runtime on Kaggle P100 / T4: ~10-15 minutes.

After download, place all files in c:/Users/dhruv/Code/hawkeye-idea/artifacts/.
The HAWKEYE backend's EmbeddingService loads the encoder + embeddings and
combines them with the LightGBM blend for new-bank cold-start scoring.

REQUIREMENTS:
  pip install torch>=2.1 pandas numpy scikit-learn pyarrow

## CELL 1 — IMPORTS & PATHS

In [ ]:
import os, sys, gc, json, time
from glob import glob

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.model_selection import train_test_split

print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} | "
      f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# Discover dataset paths recursively (handles competition + dataset mount layouts)
BASE = None
for p in sorted(glob('/kaggle/input/**/train_labels.parquet', recursive=True)):
    BASE = os.path.dirname(p)
    break
if BASE is None:
    raise FileNotFoundError("Couldn't find train_labels.parquet anywhere under /kaggle/input/. Attach the RBI NFPC Phase 2 dataset.")

ARTIFACTS_IN = None
for p in sorted(glob('/kaggle/input/**/account_feature_matrix.parquet', recursive=True)):
    parent = os.path.dirname(p)
    if os.path.exists(f'{parent}/feature_config.json'):
        ARTIFACTS_IN = parent
        break
if ARTIFACTS_IN is None:
    raise FileNotFoundError(
        "Couldn't find account_feature_matrix.parquet + feature_config.json.\n"
        "Attach the hawkeye_artifacts dataset (output of ml-idea.ipynb) to this notebook.")

print(f"Dataset:   {BASE}")
print(f"Artifacts: {ARTIFACTS_IN}")

OUT = '/kaggle/working/hawkeye_simclr'
os.makedirs(OUT, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
np.random.seed(SEED); torch.manual_seed(SEED)

## CELL 2 — LOAD FEATURE MATRIX + LABELS

In [ ]:
print("\n[1/5] Loading feature matrix + labels...")
t0 = time.time()

with open(f'{ARTIFACTS_IN}/feature_config.json') as f:
    cfg = json.load(f)
feat_clean = cfg['feat_clean']    # 105 features the M1 LightGBM uses
print(f"  Using feat_clean: {len(feat_clean)} features")

feat_df = pd.read_parquet(f'{ARTIFACTS_IN}/account_feature_matrix.parquet',
                     columns=['account_id', 'is_mule'] + feat_clean)
print(f"  Feature matrix: {feat_df.shape}")

# Z-score normalise (saved scaler exported for serve-time)
X_raw = feat_df[feat_clean].fillna(0).astype(np.float32).values
mean  = X_raw.mean(axis=0)
std   = X_raw.std(axis=0) + 1e-6
X     = (X_raw - mean) / std
y     = feat_df['is_mule'].values  # NaN for test rows
labeled_mask = ~np.isnan(y)

print(f"  Labeled accounts: {labeled_mask.sum():,} | Mules: {int(np.nansum(y)):,}")

# Save the scaler (backend needs to apply the same z-score)
with open(f'{OUT}/feat_clean_scaler.json', 'w') as f:
    json.dump({
        'feat_clean': feat_clean,
        'mean': mean.tolist(),
        'std':  std.tolist(),
    }, f, indent=2)

## CELL 3 — DATASET WITH TWO RANDOM AUGMENTED VIEWS PER ANCHOR

In [ ]:
class SimCLRDataset(Dataset):
    '''Two augmented views per anchor.
    v2: stronger augs to break the trivial-collapse pathology that left
    loss stuck at ~log(2N). Adds mixup-style blending with a random row.
    '''
    def __init__(self, X, dropout_p=0.4, noise_sigma=0.3, mixup_alpha=0.2):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.dropout_p = dropout_p
        self.noise_sigma = noise_sigma
        self.mixup_alpha = mixup_alpha
        self.n = self.X.size(0)

    def __len__(self):
        return self.n

    def _augment(self, x):
        # Random feature dropout (stronger)
        mask = (torch.rand_like(x) > self.dropout_p).float()
        x = x * mask
        # Gaussian noise (stronger)
        x = x + torch.randn_like(x) * self.noise_sigma
        # Mixup blend with a random other row (small alpha so semantics survive)
        if self.mixup_alpha > 0:
            j = torch.randint(0, self.n, (1,)).item()
            lam = float(np.random.beta(self.mixup_alpha, self.mixup_alpha))
            lam = max(lam, 1.0 - lam)  # keep most of the anchor
            x = lam * x + (1 - lam) * self.X[j]
        return x

    def __getitem__(self, idx):
        x = self.X[idx]
        return self._augment(x), self._augment(x)

ds = SimCLRDataset(X)
loader = DataLoader(ds, batch_size=512, shuffle=True,
                     num_workers=2, drop_last=True, pin_memory=torch.cuda.is_available())
print(f"\n[2/5] Dataset built: {len(ds):,} accounts | "
      f"~{len(loader)} batches/epoch @ 512 | augs: dropout=0.4, noise=0.3, mixup=0.2")

## CELL 4 — MODEL: ENCODER + PROJECTION HEAD

In [ ]:
class Encoder(nn.Module):
    def __init__(self, in_dim, embed_dim=128, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.ELU(),
            nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.ELU(),
            nn.Linear(hidden, embed_dim),
        )
    def forward(self, x):
        return self.net(x)


class ProjectionHead(nn.Module):
    '''Used for the contrastive loss only, dropped at inference.'''
    def __init__(self, in_dim=128, out_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ELU(),
            nn.Linear(in_dim, out_dim),
        )
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)


def nt_xent_loss(z1, z2, tau=0.2):
    '''v2: tau lowered from 0.5 -> 0.2 so the contrastive task is harder
    and the model can't trivially shrink all embeddings together.
    '''
    B = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    sim = torch.mm(z, z.T) / tau
    mask = torch.eye(2*B, device=z.device, dtype=torch.bool)
    sim.masked_fill_(mask, float('-inf'))
    pos_idx = torch.arange(2*B, device=z.device)
    pos_idx = (pos_idx + B) % (2*B)
    return F.cross_entropy(sim, pos_idx)


IN_DIM    = X.shape[1]
EMBED_DIM = 128
PROJ_DIM  = 64
EPOCHS    = 100   # was 50 — extra room to escape collapse
LR        = 1e-3

encoder = Encoder(IN_DIM, EMBED_DIM).to(device)
proj    = ProjectionHead(EMBED_DIM, PROJ_DIM).to(device)
opt     = torch.optim.AdamW(
    list(encoder.parameters()) + list(proj.parameters()),
    lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)

print(f"\n[3/5] Encoder: {IN_DIM} -> {EMBED_DIM} | "
      f"Projection: {EMBED_DIM} -> {PROJ_DIM} | tau=0.2")
print(f"  Trainable params: encoder={sum(p.numel() for p in encoder.parameters()):,} | "
      f"proj={sum(p.numel() for p in proj.parameters()):,}")

## CELL 5 — TRAIN

In [ ]:
print(f"\n[4/5] Training SimCLR ({EPOCHS} epochs)...")
encoder.train(); proj.train()
losses = []

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for v1, v2 in loader:
        v1, v2 = v1.to(device), v2.to(device)
        z1 = proj(encoder(v1))
        z2 = proj(encoder(v2))
        loss = nt_xent_loss(z1, z2)
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(encoder.parameters()) + list(proj.parameters()), 1.0)
        opt.step()
        epoch_loss += loss.item()
    scheduler.step()
    epoch_loss /= len(loader)
    losses.append(epoch_loss)
    if (epoch + 1) % 5 == 0:
        print(f"  epoch {epoch+1:3d}/{EPOCHS} | loss {epoch_loss:.4f} | lr {opt.param_groups[0]['lr']:.2e}")

## CELL 6 — EVAL: train logistic regression on encoder embeddings

In [ ]:
print(f"\n[5/5] Evaluating embeddings: train logistic regression on labeled subset...")

# Generate embeddings for ALL accounts
encoder.eval()
with torch.no_grad():
    embeds = []
    for i in range(0, len(X), 4096):
        batch = torch.tensor(X[i:i+4096], dtype=torch.float32, device=device)
        embeds.append(encoder(batch).cpu().numpy())
    embeds = np.concatenate(embeds, axis=0)  # (n, EMBED_DIM)

print(f"  Embeddings: {embeds.shape}")

# Linear probe — strict measure of how useful these embeddings are
y_lab = y[labeled_mask].astype(int)
X_lab = embeds[labeled_mask]
X_tr, X_va, y_tr, y_va = train_test_split(
    X_lab, y_lab, test_size=0.2, stratify=y_lab, random_state=SEED)

lr = LogisticRegression(max_iter=2000, class_weight='balanced',
                        C=1.0, random_state=SEED, n_jobs=-1)
lr.fit(X_tr, y_tr)
p_va = lr.predict_proba(X_va)[:, 1]
auc  = roc_auc_score(y_va, p_va)
f1   = f1_score(y_va, (p_va >= 0.5).astype(int))

print(f"  Linear probe on SimCLR embeddings:  AUC={auc:.5f}  F1@0.5={f1:.5f}")

# Few-shot evaluation: how good are the embeddings with very few labels?
print("\n  Few-shot probe (simulating cold-start with 50 / 200 / 500 labels):")
fewshot = {}
from sklearn.model_selection import StratifiedShuffleSplit
for n_lab in [50, 200, 500]:
    if n_lab >= len(y_tr): continue
    try:
        sss = StratifiedShuffleSplit(n_splits=1, train_size=n_lab, random_state=SEED)
        ids, _ = next(sss.split(X_tr, y_tr))
        lr2 = LogisticRegression(max_iter=2000, class_weight='balanced',
                                  C=1.0, random_state=SEED)
        lr2.fit(X_tr[ids], y_tr[ids])
        p_few = lr2.predict_proba(X_va)[:, 1]
        a_few = roc_auc_score(y_va, p_few)
        fewshot[n_lab] = float(a_few)
        print(f"    n={n_lab:>4d} labels  ->  AUC {a_few:.5f}")
    except ValueError as e:
        print(f"    n={n_lab:>4d} labels  ->  skipped ({e})")
        fewshot[n_lab] = None

## CELL 7 — EXPORT

In [ ]:
print(f"\n[6/6] Exporting to {OUT}/...")

# Save encoder weights only (drop projection head)
torch.save({
    'state_dict': encoder.state_dict(),
    'in_dim':     IN_DIM,
    'embed_dim':  EMBED_DIM,
    'feat_clean': feat_clean,
}, f'{OUT}/simclr_encoder.pt')

# Save embeddings parquet (key by account_id)
emb_df = pd.DataFrame(embeds, columns=[f'simclr_e{i:03d}' for i in range(EMBED_DIM)])
emb_df.insert(0, 'account_id', feat_df['account_id'].values)
emb_df.to_parquet(f'{OUT}/simclr_embeddings.parquet',
                   compression='zstd', index=False)

with open(f'{OUT}/simclr_metadata.json', 'w') as f:
    json.dump({
        'version':       'HAWKEYE_SIMCLR_v1',
        'in_dim':        IN_DIM,
        'embed_dim':     EMBED_DIM,
        'epochs':        EPOCHS,
        'batch_size':    512,
        'temperature':   0.5,
        'final_loss':    float(losses[-1]),
        'linear_probe_auc':  float(auc),
        'linear_probe_f1':   float(f1),
        'fewshot_auc':       fewshot,
        'training_seconds':  time.time() - t0,
    }, f, indent=2)

print(f"\n  ✓ Files in {OUT}/:")
for f in sorted(os.listdir(OUT)):
    sz = os.path.getsize(f'{OUT}/{f}') / 1024 / 1024
    print(f"    {f:<35} {sz:>8.2f} MB")

print(f"\n  Linear probe AUC: {auc:.5f}")
print(f"  Few-shot AUC: {fewshot}")
print(f"  Total time: {(time.time()-t0)/60:.1f} min")
import shutil
zip_path = shutil.make_archive('/kaggle/working/hawkeye_simclr', 'zip', OUT)
print(f"\n  Zip ready: {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")
print("  Download /kaggle/working/hawkeye_simclr.zip - unzip - drop into artifacts/")